## Data Partitioning with Azure Databricks

### Loading CSV File into Unity Catalog Volume

In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS data_modelling

In [ ]:
import requests

# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/default/data_modelling"

# List of files to download
files = ["data_partitioning_demo.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DP-750/refs/heads/main/Data_Modelling/Data/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

### Loading Data into a Dataframe

In [ ]:
df = spark.read.load(f'/Volumes/{catalog_name}/default/data_modelling/data_partitioning_demo.csv', format='csv', header=True)
display(df.limit(10))

### Saving as Table with Partitioned by Year

In [ ]:
df.write.format("delta").partitionBy("signup_year").saveAsTable("default.customers_partitioned")

### Query the Partitioned Table

In [ ]:
%sql
SELECT *
FROM customers_partitioned
WHERE signup_year = 2022;

In [ ]:
%sql
SELECT customer_id, full_name, city, balance
FROM customers_partitioned
WHERE signup_year = 2022
  AND segment = 'Retail';

In [ ]:
%sql
SELECT *
FROM customers_partitioned
WHERE signup_year IN (2021, 2022);